# 07 - Improve With Ablations And Failure Analysis

Compares router/retrieval variants to understand which design choices matter. The default experiments are offline: top-k sweeps, expected-source versus predicted-source retrieval, and base versus challenging router behavior.


## Learning Goal

Use evals as an improvement loop. This lab compares variants, sweeps `top_k`, contrasts expected-source versus predicted-source retrieval, and inspects failures so students can decide what to change next.

## Where This Fits

Progression: data sanity -> router eval -> retrieval eval -> cascade eval -> answer quality -> full benchmark -> ablation.

This is the capstone notebook. It turns eval outputs into engineering decisions: which variant helped, which slice regressed, and which failure mode deserves the next iteration?

## Related AI Evals Concepts

- Error Analysis: inspect failures instead of only reading aggregate scores.
- Synthetic Data: challenging examples stress source-selection behavior beyond the easy base set.
- AI Eval Mistakes: perfect or saturated scores should lead to harder test cases, not celebration.
- Deploy Evals: exported summaries make repeated comparisons possible.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

PROJECT_ROOT


PosixPath('/Users/christoszigkolis/Documents/Projects/simple_agentic_rag')

In [2]:
import asyncio
import importlib
import os
from typing import Any

import chromadb
import pandas as pd

import agentic_rag.ingestion as ingestion_module  # noqa: E402
import agentic_rag.retrievers as retrievers_module  # noqa: E402
import agentic_rag.router as router_module  # noqa: E402
from agentic_rag.constants import SourceType  # noqa: E402
from agentic_rag.evaluation import parse_doc_ids, score_retrieval  # noqa: E402

importlib.reload(ingestion_module)
importlib.reload(retrievers_module)
importlib.reload(router_module)

from agentic_rag.ingestion import ensure_chroma_collections  # noqa: E402
from agentic_rag.llm import OpenAITextGenerator  # noqa: E402
from agentic_rag.retrievers import ChromaRetriever  # noqa: E402
from agentic_rag.router import QueryRouter  # noqa: E402
from agentic_rag.settings import Settings  # noqa: E402
from agentic_rag.telemetry import configure_tracing  # noqa: E402

pd.set_option("display.max_colwidth", 180)


In [3]:
TRACE_DIR = PROJECT_ROOT / "otel_traces"
TRACE_DIR.mkdir(parents=True, exist_ok=True)
TRACE_FILE = TRACE_DIR / "07_experiments_and_ablation.jsonl"

trace_settings = Settings(
    _env_file=None,
    OTEL_TRACING_ENABLED=True,
    OTEL_TRACES_EXPORTER="file",
    OTEL_TRACES_FILE=TRACE_FILE,
    OTEL_SERVICE_NAME="agentic-rag-notebooks",
)
configure_tracing(trace_settings)

TRACE_FILE


PosixPath('/Users/christoszigkolis/Documents/Projects/simple_agentic_rag/otel_traces/07_experiments_and_ablation.jsonl')

## Load Experiment Test Sets


In [4]:
base_df = pd.read_csv(PROJECT_ROOT / "datasets/evaluation_dataset.csv")
challenge_df = pd.read_csv(PROJECT_ROOT / "datasets/challenging_router_evaluation_dataset.csv")

if "Query" in base_df.columns:
    base_df = base_df.rename(columns={"Query": "query", "Expected_Source_Type": "expected_source_type"})

base_df["dataset"] = "base"
challenge_df["dataset"] = "challenging"
base_df["expected_source_type"] = base_df["expected_source_type"].astype(str)
challenge_df["expected_source_type"] = challenge_df["expected_source_type"].astype(str)

datasets = pd.concat([base_df, challenge_df], ignore_index=True, sort=False)
datasets[["dataset", "query", "expected_source_type"]].head()


,dataset,query,expected_source_type
0,base,How many people are affected by X-linked chondrodysplasia punctata 1 ?,Retrieve_QnA
1,base,What are the treatments for Kawasaki disease ?,Retrieve_QnA
2,base,What are the genetic changes related to Ellis-van Creveld syndrome ?,Retrieve_QnA
3,base,What are the symptoms of Renal dysplasia-limb defects syndrome ?,Retrieve_QnA
4,base,What is (are) Fraser syndrome ?,Retrieve_QnA


In [5]:
qna_df = pd.read_csv(PROJECT_ROOT / "datasets/medical_qna_dataset.csv")
device_df = pd.read_csv(PROJECT_ROOT / "datasets/medical_device_manuals_dataset.csv")

settings = Settings(_env_file=None, chroma_path=PROJECT_ROOT / "chroma_db")
client = chromadb.PersistentClient(path=str(settings.chroma_path))

collection_summary = ensure_chroma_collections(client, qna_df, device_df)
collection_summary


,collection,exists,count
0,medical_qna,True,16407
1,medical_device_manual,True,2694


The CSV `expected_doc_ids` column stores local row numbers from the generated eval file, while Chroma returns collection document IDs such as `qna-7354` and `device-346`. Experiment retrieval metrics compare IDs by exact string match, so this notebook resolves each gold row back to the Chroma document ID before scoring.


In [6]:
LOCAL_SOURCES = {SourceType.RETRIEVE_QNA.value, SourceType.RETRIEVE_DEVICE.value}


def normalize_text(value: object) -> str:
    return " ".join(str(value or "").lower().split())


qna_id_by_question_answer = {
    (normalize_text(row["Question"]), normalize_text(row["Answer"])): f"qna-{row_index}"
    for row_index, row in qna_df.reset_index(drop=True).iterrows()
}
qna_ids_by_question = qna_df.reset_index(drop=True).groupby(
    qna_df["Question"].map(normalize_text)
).apply(lambda rows: [f"qna-{row_index}" for row_index in rows.index]).to_dict()

device_answer_columns = ["Indications_for_Use", "Contraindications", "Patient_Population"]
device_lookup_rows = []
for row_index, row in device_df.reset_index(drop=True).iterrows():
    for column in device_answer_columns:
        if column in row and not pd.isna(row[column]):
            device_lookup_rows.append(
                {
                    "doc_id": f"device-{row_index}",
                    "answer": normalize_text(row[column]),
                    "device_name": normalize_text(row["Device_Name"]),
                    "model_number": normalize_text(row["Model_Number"]),
                }
            )


def local_source_from_value(value: str) -> SourceType | None:
    try:
        source = SourceType(value)
    except ValueError:
        return None
    return source if source.value in LOCAL_SOURCES else None


def resolve_expected_doc_ids(row: pd.Series) -> list[str]:
    source = local_source_from_value(str(row["expected_source_type"]))
    query = normalize_text(row["query"])
    expected_answer = normalize_text(row.get("expected_answer", ""))

    if source == SourceType.RETRIEVE_QNA:
        exact_match = qna_id_by_question_answer.get((query, expected_answer))
        if exact_match:
            return [exact_match]
        return qna_ids_by_question.get(query, [])

    if source == SourceType.RETRIEVE_DEVICE:
        candidates = [item for item in device_lookup_rows if item["answer"] == expected_answer]
        model_matches = [item for item in candidates if item["model_number"] and item["model_number"] in query]
        if model_matches:
            return sorted({item["doc_id"] for item in model_matches})
        named_matches = [item for item in candidates if item["device_name"] and item["device_name"] in query]
        if named_matches:
            return sorted({item["doc_id"] for item in named_matches})
        return sorted({item["doc_id"] for item in candidates})

    return []


def expected_ids_from_row(row: pd.Series) -> list[str]:
    resolved_ids = resolve_expected_doc_ids(row)
    return resolved_ids or parse_doc_ids(row.get("expected_doc_ids", ""))


## Define Variant Metrics


In [7]:
LABELS = [source.value for source in SourceType]
LOCAL_SOURCES = {SourceType.RETRIEVE_QNA.value, SourceType.RETRIEVE_DEVICE.value}
settings = Settings(_env_file=None, chroma_path=PROJECT_ROOT / "chroma_db")


def source_or_none(value: str) -> SourceType | None:
    try:
        return SourceType(value)
    except ValueError:
        return None


def is_local_source(value: str) -> bool:
    return value in LOCAL_SOURCES


def lexical_score(question: str, answer: str) -> float:
    q_tokens = {token.lower().strip(".,?!:;()[]{}\"'") for token in question.split() if len(token) > 3}
    a_tokens = {token.lower().strip(".,?!:;()[]{}\"'") for token in answer.split() if len(token) > 3}
    if not q_tokens or not a_tokens:
        return 0.0
    return len(q_tokens & a_tokens) / len(q_tokens)


def preview_text(text: str, max_words: int = 50) -> str:
    return " ".join(text.split()[:max_words])

async def experiment_row(row: pd.Series, top_k: int, retrieval_policy: str) -> dict[str, Any]:
    query = str(row["query"])
    expected_source = str(row["expected_source_type"])
    expected_ids = expected_ids_from_row(row)
    router = QueryRouter(mode="heuristic")
    predicted_source = (await router.route(query)).value
    retrieval_source = expected_source if retrieval_policy == "expected_source" else predicted_source

    docs = []
    if is_local_source(retrieval_source):
        retriever = ChromaRetriever(chroma_path=str(settings.chroma_path), top_k=top_k)
        docs = await retriever.retrieve(SourceType(retrieval_source), query)

    retrieved_ids = [doc.doc_id for doc in docs]
    retrieval_score = score_retrieval(retrieved_ids, expected_ids)
    has_gold = bool(expected_ids)
    return {
        "dataset": row["dataset"],
        "query": query,
        "expected_source_type": expected_source,
        "predicted_source_type": predicted_source,
        "route_correct": predicted_source == expected_source,
        "retrieval_policy": retrieval_policy,
        "retrieval_source": retrieval_source,
        "top_k": top_k,
        "expected_doc_ids": "|".join(expected_ids),
        "retrieved_doc_ids": "|".join(retrieved_ids),
        "precision_at_k": retrieval_score.precision_at_k if has_gold else None,
        "recall_at_k": retrieval_score.recall_at_k if has_gold else None,
        "hit_at_k": retrieval_score.hit_at_k if has_gold else None,
        "mrr": retrieval_score.mrr if has_gold else None,
        "metric_status": "scored" if has_gold else ("qualitative_only_no_gold_doc_ids" if is_local_source(expected_source) else "skipped_web_source"),
    }


async def run_experiments(df: pd.DataFrame, top_k_values: list[int], retrieval_policies: list[str]) -> pd.DataFrame:
    rows = []
    for top_k in top_k_values:
        for policy in retrieval_policies:
            rows.extend(await asyncio.gather(*(experiment_row(row, top_k, policy) for _, row in df.iterrows())))
    return pd.DataFrame(rows)


def summarize_experiments(results: pd.DataFrame) -> pd.DataFrame:
    scored = results[results["metric_status"] == "scored"]
    if scored.empty:
        return pd.DataFrame()
    return scored.groupby(["dataset", "retrieval_policy", "top_k"], dropna=False).agg(
        examples=("query", "count"),
        router_accuracy=("route_correct", "mean"),
        precision_at_k=("precision_at_k", "mean"),
        recall_at_k=("recall_at_k", "mean"),
        hit_at_k=("hit_at_k", "mean"),
        mrr=("mrr", "mean"),
    ).reset_index()


## Compare Variants With Offline Ablations


In [8]:
TOP_K_VALUES = [1, 3, 5, 10]
RETRIEVAL_POLICIES = ["expected_source", "predicted_source"]

base_experiment_results = await run_experiments(base_df, TOP_K_VALUES, RETRIEVAL_POLICIES)
challenge_experiment_results = await run_experiments(challenge_df, TOP_K_VALUES, RETRIEVAL_POLICIES)

base_experiment_results.head()


,dataset,query,expected_source_type,predicted_source_type,route_correct,retrieval_policy,retrieval_source,top_k,expected_doc_ids,retrieved_doc_ids,precision_at_k,recall_at_k,hit_at_k,mrr,metric_status
0,base,How many people are affected by X-linked chondrodysplasia punctata 1 ?,Retrieve_QnA,Retrieve_QnA,True,expected_source,Retrieve_QnA,1,qna-7354,qna-7354,1.0,1.0,1.0,1.0,scored
1,base,What are the treatments for Kawasaki disease ?,Retrieve_QnA,Retrieve_QnA,True,expected_source,Retrieve_QnA,1,qna-6837,qna-12125,0.0,0.0,0.0,0.0,scored
2,base,What are the genetic changes related to Ellis-van Creveld syndrome ?,Retrieve_QnA,Retrieve_QnA,True,expected_source,Retrieve_QnA,1,qna-8175,qna-8175,1.0,1.0,1.0,1.0,scored
3,base,What are the symptoms of Renal dysplasia-limb defects syndrome ?,Retrieve_QnA,Retrieve_QnA,True,expected_source,Retrieve_QnA,1,qna-14533,qna-15015,0.0,0.0,0.0,0.0,scored
4,base,What is (are) Fraser syndrome ?,Retrieve_QnA,Retrieve_QnA,True,expected_source,Retrieve_QnA,1,qna-9958,qna-15453,0.0,0.0,0.0,0.0,scored


In [9]:
experiment_results = pd.concat([base_experiment_results, challenge_experiment_results], ignore_index=True)
experiment_summary = summarize_experiments(experiment_results)
experiment_summary


,dataset,retrieval_policy,top_k,examples,router_accuracy,precision_at_k,recall_at_k,hit_at_k,mrr
0,base,expected_source,1,24,1.0,0.166667,0.166667,0.166667,0.166667
1,base,expected_source,3,24,1.0,0.097222,0.291667,0.291667,0.229167
2,base,expected_source,5,24,1.0,0.066667,0.333333,0.333333,0.239583
3,base,expected_source,10,24,1.0,0.041667,0.416667,0.416667,0.248958
4,base,predicted_source,1,24,1.0,0.166667,0.166667,0.166667,0.166667
5,base,predicted_source,3,24,1.0,0.097222,0.291667,0.291667,0.229167
6,base,predicted_source,5,24,1.0,0.066667,0.333333,0.333333,0.239583
7,base,predicted_source,10,24,1.0,0.041667,0.416667,0.416667,0.248958


## Measure How Routing Errors Affect Retrieval


In [10]:
base_policy_delta = experiment_summary[experiment_summary["dataset"] == "base"].pivot_table(
    index="top_k",
    columns="retrieval_policy",
    values="hit_at_k",
)
base_policy_delta["predicted_minus_expected"] = base_policy_delta.get("predicted_source") - base_policy_delta.get("expected_source")
base_policy_delta


retrieval_policy,expected_source,predicted_source,predicted_minus_expected
top_k,,,
1,0.166667,0.166667,0.0
3,0.291667,0.291667,0.0
5,0.333333,0.333333,0.0
10,0.416667,0.416667,0.0


## Inspect Failure Cases From Hard Examples


In [11]:
challenge_router_failures = challenge_experiment_results[
    (challenge_experiment_results["top_k"] == 3)
    & (challenge_experiment_results["retrieval_policy"] == "predicted_source")
    & (~challenge_experiment_results["route_correct"])
][["query", "expected_source_type", "predicted_source_type", "retrieval_source", "metric_status", "retrieved_doc_ids"]]

challenge_router_failures


,query,expected_source_type,predicted_source_type,retrieval_source,metric_status,retrieved_doc_ids
48,"Does the Pro127 Electrosurgical Unit support pediatric use, and what symptoms would indicate a complication?",Retrieve_Device,Retrieve_QnA,Retrieve_QnA,qualitative_only_no_gold_doc_ids,qna-12202|qna-8253|qna-12513
54,"If the retrieved Q&A answer about dystonia is irrelevant, should the graph fall back to Tavily?",Web_Search,Retrieve_QnA,Retrieve_QnA,skipped_web_source,qna-4078|qna-11610|qna-4100
56,Are there recent safety alerts for the manufacturer Abbott's electrosurgical units?,Web_Search,Retrieve_Device,Retrieve_Device,skipped_web_source,device-2318|device-754|device-2015
59,"Who is at risk for LCM, and has there been a recent outbreak near Athens?",Web_Search,Retrieve_QnA,Retrieve_QnA,skipped_web_source,qna-3840|qna-2312|qna-140


## Pay Attention To

- Ablations should answer a concrete question, such as whether larger `top_k` improves recall enough to justify more context.
- Compare base and challenging slices separately; averaging them can hide important behavior.
- Expected-source retrieval is a control, while predicted-source retrieval reflects the actual routed system.
- The next product change should come from failure inspection, not just the largest aggregate score.


## Optional Advanced Path: LLM Router Evaluation

This section is disabled by default because it makes API calls. Set `RUN_LLM_ROUTER = True` to compare the LLM router against both the base and challenging router datasets. It reads `OPENAI_API_KEY` from the active environment only; it does not load `.env`.


In [12]:
RUN_LLM_ROUTER = True
LLM_ROUTER_MODEL = os.getenv("OPENAI_ROUTER_MODEL", "gpt-5-nano")
LLM_ROUTER_TIMEOUT = float(os.getenv("OPENAI_TIMEOUT", "60"))

llm_router_results = pd.DataFrame()
llm_router_summary = pd.DataFrame()


def load_router_eval_dataset(path: Path, dataset_name: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    if "Query" in df.columns:
        df = df.rename(columns={"Query": "query", "Expected_Source_Type": "expected_source_type"})
    df = df.copy()
    df["dataset"] = dataset_name
    df["expected_source_type"] = df["expected_source_type"].astype(str)
    return df


async def evaluate_llm_router_dataset(router: QueryRouter, df: pd.DataFrame) -> pd.DataFrame:
    async def route_row(query: str) -> str:
        return (await router.route(query)).value

    predictions = await asyncio.gather(*(route_row(str(query)) for query in df["query"].tolist()))
    results = df.copy()
    results["router_mode"] = "llm"
    results["predicted_source_type"] = predictions
    results["route_correct"] = results["expected_source_type"] == results["predicted_source_type"]
    keep_columns = [
        "dataset",
        "router_mode",
        "query",
        "expected_source_type",
        "predicted_source_type",
        "route_correct",
        "category",
        "rationale",
    ]
    return results[[column for column in keep_columns if column in results.columns]]


def summarize_llm_router_results(results: pd.DataFrame) -> pd.DataFrame:
    if results.empty:
        return pd.DataFrame()
    return results.groupby(["dataset", "router_mode"], dropna=False).agg(
        examples=("query", "count"),
        accuracy=("route_correct", "mean"),
        failures=("route_correct", lambda values: int((~values).sum())),
    ).reset_index()


if RUN_LLM_ROUTER:
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        raise RuntimeError("OPENAI_API_KEY must be set in the active environment for LLM router evaluation")

    llm_router = QueryRouter(
        generator=OpenAITextGenerator(api_key=api_key, model=LLM_ROUTER_MODEL, timeout=LLM_ROUTER_TIMEOUT),
        mode="llm",
    )
    llm_router_base_df = load_router_eval_dataset(PROJECT_ROOT / "datasets/evaluation_dataset.csv", "base")
    llm_router_challenge_df = load_router_eval_dataset(PROJECT_ROOT / "datasets/challenging_router_evaluation_dataset.csv", "challenging")
    llm_router_results = pd.concat(
        [
            await evaluate_llm_router_dataset(llm_router, llm_router_base_df),
            await evaluate_llm_router_dataset(llm_router, llm_router_challenge_df),
        ],
        ignore_index=True,
    )
    llm_router_summary = summarize_llm_router_results(llm_router_results)

llm_router_summary


,dataset,router_mode,examples,accuracy,failures
0,base,llm,27,1.000000,0
1,challenging,llm,15,0.866667,2


In [13]:
if not llm_router_results.empty:
    display(llm_router_summary)
    display(pd.crosstab(
        [llm_router_results["dataset"], llm_router_results["expected_source_type"]],
        llm_router_results["predicted_source_type"],
        dropna=False,
    ))
    display(llm_router_results.loc[~llm_router_results["route_correct"]])


,dataset,router_mode,examples,accuracy,failures
0,base,llm,27,1.000000,0
1,challenging,llm,15,0.866667,2


predicted_source_type             Retrieve_Device  Retrieve_QnA  Web_Search
dataset     expected_source_type                                           
base        Retrieve_Device                    12             0           0
            Retrieve_QnA                        0            12           0
            Web_Search                          0             0           3
challenging Retrieve_Device                     6             1           0
            Retrieve_QnA                        0             1           0
            Web_Search                          0             1           6

,dataset,router_mode,query,expected_source_type,predicted_source_type,route_correct,category,rationale
27,challenging,llm,The manual says the Model 1606 Electrosurgical Unit is contraindicated near MRI environments; what general risks do MRI environments create for implanted devices?,Retrieve_Device,Retrieve_QnA,False,device_plus_general_medical,"Mentions general risks, but the answer depends on a device manual contraindication."
28,challenging,llm,"For a child with fever and rash after vaccination, should I use the Q&A knowledge base or search current outbreak updates?",Web_Search,Retrieve_QnA,False,explicit_source_meta_question,Asks about source selection and current outbreak updates rather than a stable disease answer.


## Export Experiment Artifacts


In [14]:
results_path = PROJECT_ROOT / "output/experiments_ablation_results.csv"
summary_path = PROJECT_ROOT / "output/experiments_ablation_summary.csv"
results_path.parent.mkdir(parents=True, exist_ok=True)
summary_path.parent.mkdir(parents=True, exist_ok=True)
experiment_results.to_csv(results_path, index=False)
experiment_summary.to_csv(summary_path, index=False)

results_path, summary_path


(PosixPath('/Users/christoszigkolis/Documents/Projects/simple_agentic_rag/output/experiments_ablation_results.csv'),
 PosixPath('/Users/christoszigkolis/Documents/Projects/simple_agentic_rag/output/experiments_ablation_summary.csv'))